# Single-Spin Metropolis algorithm (checkerboard) on 2D Ising Model

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit
from tqdm import tqdm
import os

In [2]:
@njit
def initial_state(L, random_start=True):
    state = np.empty((L, L), dtype=np.int8)
    if random_start:
        # Random ±1 using threshold
        for i in range(L):
            for j in range(L):
                if np.random.random() < 0.5:
                    state[i, j] = 1
                else:
                    state[i, j] = -1
    else:
        # All spins up (+1)
        state[:] = 1
    return state

In [3]:
@njit(cache=True)
def calc_total_energy(state, J, h):
    # Only used once at the start to initialize the counter
    H = 0.0
    rows, cols = state.shape
    for i in range(rows):
        for j in range(cols):
            s = state[i, j]
            neighbor_sum = state[i, (j + 1) % cols] + state[(i + 1) % rows, j]
            H -= J * s * neighbor_sum
            H -= h * s
    return H

In [4]:
@njit(cache=True)
def get_magnetization(state):
    return np.sum(state)

In [5]:
def compute_susceptibility(Ms, beta, N):
    m = Ms / N
    m2_mean = np.mean(m**2)
    m_mean = np.mean(np.abs(m))
    return beta * N * (m2_mean - m_mean**2)

In [6]:
@njit(parallel=True, cache=True)
def run_simulation_incremental(state, J, h, beta, n_sweeps, record_interval, initial_E):
    """
    Runs simulation tracking Energy changes (dE) instead of recalculating 
    Total Energy. This allows for recording every single step with 
    almost zero performance penalty.
    """
    rows, cols = state.shape
    current_E = initial_E
    
    # Precompute Lookup Table for exp values: saves computation time
    # Indices 0 to 4 map to s*sum values -4, -2, 0, 2, 4
    exp_table = np.zeros(5, dtype=np.float64)
    possible_vals = np.array([-4, -2, 0, 2, 4], dtype=np.float64)
    
    # Lookup table for the dE values themselves to update current_E fast
    dE_vals = np.zeros(5, dtype=np.float64) 
    
    for k in range(5):
        val = possible_vals[k]
        dE = 2.0 * J * val + 2.0 * h
        dE_vals[k] = dE
        if dE <= 0:
            exp_table[k] = 2.0 
        else:
            exp_table[k] = np.exp(-beta * dE)

    n_records = n_sweeps // record_interval
    energies = np.zeros(n_records, dtype=np.float64)
    magnetizations = np.zeros(n_records, dtype=np.float64)
    rec_idx = 0
    
    # Note: Parallelizing the Checkerboard loop while tracking a scalar (current_E) is tricky because of race conditions. 
    # STRICTLY SPEAKING: To track E exactly in parallel, we need atomic adds or a reduction. 
    # For MCMC assignments, it is safer/easier to run the inner spatial loops in serial if we need to track E incrementally, or accept a recalculation cost.
    
    # Since Numba parallel reduction is complex, we will run the spatial loop serial for absolute correctness of E, but it's still very fast.
    # (Removing 'parallel=True' from the decorator or the range is safer here)
    
    for sweep in range(n_sweeps):
        
        # Checkerboard update (Parity 0 then 1)
        for parity in range(2):
            for i in range(rows):
                for j in range((i + parity) % 2, cols, 2):
                    s = state[i, j]
                    neighbor_sum = (
                        state[(i + 1) % rows, j]
                        + state[(i - 1) % rows, j]
                        + state[i, (j + 1) % cols]
                        + state[i, (j - 1) % cols]
                    )
                    
                    # idx for -4 to 4 -> 0 to 4
                    idx = (s * neighbor_sum + 4) // 2
                    
                    # Check flip
                    if exp_table[idx] > 1.0 or np.random.random() < exp_table[idx]:
                        state[i, j] = -s
                        # Update energy incrementally
                        # If we flip, the energy change is exactly dE
                        current_E += dE_vals[idx]

        if (sweep + 1) % record_interval == 0:
            energies[rec_idx] = current_E
            magnetizations[rec_idx] = get_magnetization(state)
            rec_idx += 1
            
    return energies, magnetizations, state

In [7]:
def plot_state(state):
    plt.imshow(state, cmap='gray', vmin=-1, vmax=1)
    plt.axis('off')
    plt.show()

def plot_energies_and_magnetizations(energies, magnetizations):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(energies, label='Energy') 
    plt.xlabel('Step')
    plt.ylabel('Energy')
    plt.title('Energy vs. Steps')

    plt.subplot(1, 2, 2)
    plt.plot(magnetizations, label='Magnetization', color='orange')
    plt.xlabel('Step')
    plt.ylabel('Magnetization')
    plt.title('Magnetization vs. Steps')

    plt.tight_layout()
    plt.show()

## Simulation

In [8]:
base_dir = f"data/Metropolis_Temp_Scan"
os.makedirs(base_dir, exist_ok=True)

In [9]:
L_ls = [16,32,64,128]
h_ls = [0.0, -0.002, 0.002]
beta_ls = [0.1, 0.2, 0.3, 0.4, 0.41, 0.42, 0.43, 0.44, 0.4407, 0.45, 0.46, 0.47, 0.48, 0.49, 0.5, 0.6]
J = 1.0

n_sweeps = 600000

record_interval = 1

In [10]:
with open(f"{base_dir}/beta_list.txt", "w") as f:
    for beta in beta_ls:
        f.write(f"{beta}\n")

In [11]:
print(f"Estimating runtime...")
# Run a tiny warm-up to compile JIT
init_st = initial_state(L=16)
init_E = calc_total_energy(init_st, J, 0.0)
run_simulation_incremental(init_st, J, 0.0, 0.4, 100, 1, init_E)
print("JIT compiled.")

Estimating runtime...
JIT compiled.


Main simulation loop

In [12]:
for L in tqdm(L_ls, desc="Lattice sizes"):
    L_directory = f"{base_dir}/L_{L}"
    os.makedirs(L_directory, exist_ok=True)
    print(f"\n Starting simulations for L = {L}")

    for h in tqdm(h_ls, desc=f"h values for L={L}"):
        h_dir = f"{L_directory}/h_{h}"
        os.makedirs(h_dir, exist_ok=True)
        print(f"→ External field h = {h}")

        # Write parameter info file once per (L, h)
        param_path = f"{h_dir}/parameters.txt"
        with open(param_path, "w") as f:
            f.write(f"Lattice size: {L}x{L}\n")
            f.write(f"Interaction strength J: {J}\n")
            f.write(f"External field h: {h}\n")
            f.write(f"Number of sweeps (steps = sweeps * L * L): {n_sweeps}\n")
            f.write(f"Record interval: {record_interval}\n")
            f.write(f"Output directory: {h_dir}\n")
            f.write("Data saved as: mag_eng_beta_*.npz\n")
            f.write("Contents: energies, magnetizations\n")
        print(f"Parameters written to {param_path}")

        for beta in tqdm(beta_ls, desc=f"β values for L={L}, h={h}"):
            print(f"Running β = {beta:.4f} ...")

            # Initialize state for this beta
            st = initial_state(L)
            E_start = calc_total_energy(st, J, h)

            # Run simulation
            Es, Ms, final_st = run_simulation_incremental(
                st, J, h, beta, n_sweeps, record_interval, E_start
            )

            # Save results
            out_path = f"{h_dir}/mag_eng_beta_{beta:.4f}.npz"
            np.savez_compressed(out_path, energies=Es, magnetizations=Ms)
            print(f"Saved results → {out_path} [{len(Es)} records]")

        print(f"Finished all β values for h = {h}")
    print(f"Completed all external fields for L = {L}")
print("\n All simulations finished successfully!")

Lattice sizes:   0%|          | 0/4 [00:00<?, ?it/s]


 Starting simulations for L = 16


→ External field h = 0.0
Parameters written to data/Metropolis_Temp_Scan/L_16/h_0.0/parameters.txt


Running β = 0.1000 ...


Saved results → data/Metropolis_Temp_Scan/L_16/h_0.0/mag_eng_beta_0.1000.npz [600000 records]
Running β = 0.2000 ...


...

Saved results → data/Metropolis_Temp_Scan/L_128/h_0.002/mag_eng_beta_0.4800.npz [600000 records]
Running β = 0.4900 ...


Saved results → data/Metropolis_Temp_Scan/L_128/h_0.002/mag_eng_beta_0.4900.npz [600000 records]
Running β = 0.5000 ...


Saved results → data/Metropolis_Temp_Scan/L_128/h_0.002/mag_eng_beta_0.5000.npz [600000 records]
Running β = 0.6000 ...



Lattice sizes: 100%|██████████| 4/4 [7:11:22<00:00, 6470.74s/it]

Saved results → data/Metropolis_Temp_Scan/L_128/h_0.002/mag_eng_beta_0.6000.npz [600000 records]
Finished all β values for h = 0.002
Completed all external fields for L = 128

 All simulations finished successfully!
